> 📄 English: [fastapi-agent-tutorial.ipynb](fastapi-agent-tutorial.ipynb)
![](https://europe-west1-atp-views-tracker.cloudfunctions.net/working-analytics?notebook=tutorials--fastapi-agent--fastapi-agent-tutorial)

# 用 FastAPI 部署 Agent


## 什么是 FastAPI，为什么要用它部署 Agent？

FastAPI 是一个现代、高性能的 Web 框架，用于用 Python 构建 API。它于 2018 年发布，凭借速度、易用性和开发者友好的特性迅速流行起来。

FastAPI 的核心目标是创建能高效处理请求、同时提供健壮校验与文档的 REST API。对于 AI Agent 部署，FastAPI 提供了几个关键优势：

- **异步支持（Asynchronous Support）**：AI Agent 常常需要高效处理并发请求。FastAPI 原生支持 async/await，能够处理数千个并发连接，非常适合在不阻塞的情况下并行处理多个 Agent 请求。

- **流式响应（Streaming Responses）**：Agent 常常增量式地生成内容（逐个 token）。FastAPI 的流式响应能力可以在 Agent 输出生成的同时实时传输，带来更灵敏的用户体验。

- **类型校验（Type Validation）**：在与 Agent 打交道时，确保输入格式正确至关重要。FastAPI 使用 Pydantic 自动校验请求，在异常输入到达 Agent 之前就将其拦截，并提供清晰的错误消息。

- **性能（Performance）**：FastAPI 构建在 Starlette 和 Uvicorn 之上，性能接近原生。对于计算密集型的 Agent 应用，这意味着你的基础设施能高效地处理 API 开销，把更多资源留给真正的 Agent 处理。

- **自动文档（Automatic Documentation）**：当把 Agent API 暴露给多个用户或团队时，文档变得必不可少。FastAPI 通过 Swagger UI 和 ReDoc 自动生成交互式 API 文档，让其他人更容易理解和使用你的 Agent。

- **模式约束（Schema Enforcement）**：Pydantic 模型确保发给 Agent 的请求和它的响应都符合预定义的模式，使 Agent 行为更可预测，也更容易与其他系统集成。

在本教程中，我们将构建一个完整的 API，同时用同步端点和流式端点来部署一个 AI Agent，展示 FastAPI 的特性如何应对在生产中部署 Agent 所面临的具体挑战。


## 前置准备

开始之前，先安装必要的包：


In [ ]:
!pip install fastapi uvicorn pydantic

如果你打算使用流式功能，还需要安装：


In [ ]:
!pip install sse-starlette

## Agent 快速回顾

先定义一个简单的 Agent，然后通过 API 把它暴露出去。这里可以是任何 Agent 实现，不过在本教程中，我们会创建一个模拟 AI Agent 响应用户查询的基础示例：


In [ ]:
class SimpleAgent:
    def __init__(self, name="FastAPI Agent"):
        self.name = name
    
    def generate_response(self, query):
        """Generate a synchronous response to a user query"""
        return f"Agent {self.name} received: '{query}'\nResponse: This is a simulated agent response."
    
    async def generate_response_stream(self, query):
        """Generate a streaming response to a user query"""
        import asyncio
        
        prefix = f"Agent {self.name} thinking about: '{query}'\n"
        response = "This is a simulated agent response that streams token by token."
        
        # Yield the prefix as a single chunk
        yield prefix
        
        # Stream the response token by token with small delays
        for token in response.split():
            await asyncio.sleep(0.1)  # Simulate thinking time
            yield token + " "

# Test our agent directly
agent = SimpleAgent()
test_query = "Hello, what can you do?"
print(agent.generate_response(test_query))

这个简单的 Agent 既能生成同步响应，也能生成流式响应。实际使用中，你可以把它替换成更复杂的 Agent，比如微调过的 LLM、RAG 系统或其他任何 AI Agent。


## 最小化 FastAPI 应用

现在，创建一个带健康检查端点（health check）的最小化 FastAPI 应用：


In [ ]:
from fastapi import FastAPI

# Initialize FastAPI app
app = FastAPI(
    title="Agent API",
    description="A simple API that serves an AI agent",
    version="0.1.0"
)

# Create an instance of our agent
agent = SimpleAgent()

# Health check endpoint
@app.get("/health")
def health_check():
    """Check if the API is running"""
    return {"status": "ok", "message": "API is operational"}

这会创建一个带有元数据和健康检查端点的基础 FastAPI 应用。健康检查是验证 API 是否正常运行的一种简单方式。


## POST /agent - 同步端点

现在，为我们的 Agent 创建一个同步端点。我们将使用 Pydantic 模型来定义请求和响应结构：


In [ ]:
from pydantic import BaseModel
from typing import Optional

# Define request and response models
class QueryRequest(BaseModel):
    query: str
    context: Optional[str] = None
    
    class Config:
        schema_extra = {
            "example": {
                "query": "What is FastAPI?",
                "context": "I'm a beginner programmer."
            }
        }

class QueryResponse(BaseModel):
    response: str
    
    class Config:
        schema_extra = {
            "example": {
                "response": "FastAPI is a modern, high-performance web framework for building APIs with Python."
            }
        }

# Create a synchronous endpoint for the agent
@app.post("/agent", response_model=QueryResponse)
def query_agent(request: QueryRequest):
    """Get a response from the agent"""
    response = agent.generate_response(request.query)
    return QueryResponse(response=response)

这个端点接受带 JSON 请求体的 POST 请求，请求体包含一个 "query" 字段和一个可选的 "context" 字段。它返回带有 Agent 答复的 JSON 响应。


## POST /agent/stream - Token 流式输出

对于很多 AI 应用来说，token 流式输出能提供更好的用户体验。让我们实现一个流式端点：


In [ ]:
from fastapi.responses import StreamingResponse
import json

@app.post("/agent/stream")
async def stream_agent(request: QueryRequest):
    """Stream a response from the agent token by token"""
    
    async def event_generator():
        async for token in agent.generate_response_stream(request.query):
            # Format as a JSON object
            data = json.dumps({"token": token})
            yield f"data: {data}\n\n"

    return StreamingResponse(
        event_generator(),
        media_type="text/event-stream"
    )

这个端点使用服务器发送事件（SSE）逐 token 地流式返回 Agent 的响应。客户端可以在 token 到达时逐步处理它们，从而获得更具交互性的体验。

对于更完善的实现，你可能想使用 `sse-starlette` 包：


In [ ]:
from sse_starlette.sse import EventSourceResponse

@app.post("/agent/stream-sse")
async def stream_agent_sse(request: QueryRequest):
    """Stream a response using SSE with the sse-starlette package"""
    
    async def event_generator():
        async for token in agent.generate_response_stream(request.query):
            yield {"data": json.dumps({"token": token})}
    
    return EventSourceResponse(event_generator())

这是服务器发送事件（SSE）更健壮的实现方式。


## 创建完整的应用

现在，把所有内容整合成一个完整的 FastAPI 应用。在你的 `scripts` 目录下创建一个名为 `fastapi_agent.py` 的文件：


In [ ]:
from fastapi import FastAPI, Depends, HTTPException, Header
from fastapi.responses import StreamingResponse
from pydantic import BaseModel
from typing import Optional
import json
import os
import asyncio

# Define our simple agent class
class SimpleAgent:
    def __init__(self, name="FastAPI Agent"):
        self.name = name
    
    def generate_response(self, query):
        """Generate a synchronous response to a user query"""
        return f"Agent {self.name} received: '{query}'\nResponse: This is a simulated agent response."
    
    async def generate_response_stream(self, query):
        """Generate a streaming response to a user query"""
        prefix = f"Agent {self.name} thinking about: '{query}'\n"
        response = "This is a simulated agent response that streams token by token."
        
        # Yield the prefix as a single chunk
        yield prefix
        
        # Stream the response token by token with small delays
        for token in response.split():
            await asyncio.sleep(0.1)  # Simulate thinking time
            yield token + " "

# Define request and response models
class QueryRequest(BaseModel):
    query: str
    context: Optional[str] = None
    
    class Config:
        schema_extra = {
            "example": {
                "query": "What is FastAPI?",
                "context": "I'm a beginner programmer."
            }
        }

class QueryResponse(BaseModel):
    response: str
    
    class Config:
        schema_extra = {
            "example": {
                "response": "FastAPI is a modern, high-performance web framework for building APIs with Python."
            }
        }

# Initialize FastAPI app
app = FastAPI(
    title="Agent API",
    description="A simple API that serves an AI agent",
    version="0.1.0"
)

# Create an instance of our agent
agent = SimpleAgent()

# Health check endpoint
@app.get("/health")
def health_check():
    """Check if the API is running"""
    return {"status": "ok", "message": "API is operational"}

# Create a synchronous endpoint for the agent
@app.post("/agent", response_model=QueryResponse)
def query_agent(request: QueryRequest):
    """Get a response from the agent"""
    response = agent.generate_response(request.query)
    return QueryResponse(response=response)

# Create a streaming endpoint for the agent
@app.post("/agent/stream")
async def stream_agent(request: QueryRequest):
    """Stream a response from the agent token by token"""
    
    async def event_generator():
        async for token in agent.generate_response_stream(request.query):
            # Format as a JSON object
            data = json.dumps({"token": token})
            yield f"data: {data}\n\n"

    return StreamingResponse(
        event_generator(),
        media_type="text/event-stream"
    )

## 运行服务器

现在我们已经有了 FastAPI 应用，用 uvicorn 运行它：


In [ ]:
# This command should be run in the terminal, not in a notebook cell
!cd tutorials/fastapi-agent && uvicorn fastapi_agent:app --reload

`--reload` 标志会启用热重载，当你修改代码时服务器会自动重启。这在开发过程中很有帮助。

运行起来之后，你可以访问：
- API 文档：http://localhost:8000/docs
- 备选文档：http://localhost:8000/redoc
- 健康检查端点：http://localhost:8000/health


## 简单的客户端测试

让我们用一个简单的 Python 客户端来测试 API：


In [ ]:
import requests
import json

# Test the synchronous endpoint
response = requests.post(
    "http://localhost:8000/agent", 
    json={"query": "What is FastAPI?"}
)
print("Synchronous Response:")
print(response.json())
print("\n" + "-" * 40 + "\n")

# Test the streaming endpoint
response = requests.post(
    "http://localhost:8000/agent/stream",
    json={"query": "Tell me about streaming"},
    stream=True
)

print("Streaming Response:")
for line in response.iter_lines():
    if line:
        # Parse the SSE format
        line = line.decode('utf-8')
        if line.startswith('data: '):
            data = json.loads(line[6:])
            print(data["token"], end="")

这个脚本同时测试了我们 API 的同步端点和流式端点。


## 添加基础 Auth Key（可选）

在生产环境中，你可能想添加简单的 API key 认证。让我们扩展 FastAPI 应用，检查 API key：


In [ ]:
from fastapi import Depends, HTTPException, Header

# Function to validate API key
async def verify_api_key(x_api_key: str = Header(None)):
    """Verify the API key provided in the X-API-Key header"""
    # Get the API key from environment variable
    api_key = os.environ.get("API_KEY")
    
    # If no API key is set in the environment, skip validation
    if not api_key:
        return True
    
    # If API key is set but not provided in the request, return 401
    if not x_api_key:
        raise HTTPException(status_code=401, detail="API Key is missing")
    
    # If API key doesn't match, return 403
    if x_api_key != api_key:
        raise HTTPException(status_code=403, detail="Invalid API Key")
    
    return True

# Update endpoints to require API key
@app.post("/agent", response_model=QueryResponse)
def query_agent(request: QueryRequest, authorized: bool = Depends(verify_api_key)):
    """Get a response from the agent"""
    response = agent.generate_response(request.query)
    return QueryResponse(response=response)

@app.post("/agent/stream")
async def stream_agent(request: QueryRequest, authorized: bool = Depends(verify_api_key)):
    """Stream a response from the agent token by token"""
    # Same implementation as before

有了这个更新之后，如果你设置了 `API_KEY` 环境变量，API 就会要求所有请求都在 `X-API-Key` 请求头中携带匹配的 key。


## 单元测试

让我们使用 pytest 和 FastAPI 测试客户端，为我们的 FastAPI 应用编写简单的单元测试：


In [ ]:
from fastapi.testclient import TestClient
from scripts.fastapi_agent import app

client = TestClient(app)

def test_health_check():
    """Test the health check endpoint"""
    response = client.get("/health")
    assert response.status_code == 200
    assert response.json()["status"] == "ok"

def test_agent_endpoint():
    """Test the synchronous agent endpoint"""
    response = client.post(
        "/agent",
        json={"query": "Test query"}
    )
    assert response.status_code == 200
    assert "response" in response.json()
    assert "Agent" in response.json()["response"]

def test_stream_endpoint():
    """Test the streaming agent endpoint"""
    with client.stream("POST", "/agent/stream", json={"query": "Test query"}) as response:
        assert response.status_code == 200
        assert response.headers["content-type"] == "text/event-stream"
        # Check that we receive at least some content
        content = response.iter_content().read()
        assert len(content) > 0

把这些测试保存到 tests 目录下名为 `test_fastapi_agent.py` 的文件中，然后用 pytest 运行：


In [ ]:
# This command should be run in the terminal, not in a notebook cell
!pytest -xvs tests/test_fastapi_agent.py

## 下一步

现在你已经有一个运行中的基础 FastAPI Agent 服务了，下面是一些后续可以做的事：

- **添加更高级的 Agent**：把简单 Agent 替换成你生产就绪的 Agent
- **实现认证与限流**：为生产环境添加更完善的认证和限流
- **添加日志与监控中间件**：实现用于请求日志记录和性能监控的中间件
- **部署上线**：用 Docker、Kubernetes 或云服务把 FastAPI 应用部署到生产环境
- **实现异步数据库连接**：添加数据库集成，用于存储对话历史或其他数据
- **添加后台任务**：用 FastAPI 的后台任务处理长时间运行的操作


## 结语

在本教程中，我们构建了一个 FastAPI 应用，通过同步端点和流式端点来部署一个简单的 AI Agent。我们涵盖了设置 FastAPI 的基础知识、用 Pydantic 模型进行请求/响应校验、实现同步与流式端点，以及添加简单的认证。

FastAPI 集性能、自动文档和开发者友好的特性于一身，是生产环境中部署 AI Agent 的绝佳选择。遵循本教程中的模式，你就可以为自己的 AI Agent 创建健壮、生产就绪的 API。
